In [12]:
# !pip install pymupdf
import pymupdf
# !pip install pandas
import pandas as pd

In [13]:
import os
import json
import csv

In [14]:
jrb_edition = 4
jrb_folderpath = os.path.join("..", "public", f"jrb-{jrb_edition}-ed")
print(jrb_folderpath) 
os.makedirs(jrb_folderpath, exist_ok=True)

../public/jrb-4-ed


In [15]:
jrb_filepath = os.path.join("..", "public", f"jrb_{jrb_edition}.pdf")
print(jrb_filepath)

song_name_page_nums = os.path.join("..", "data_files", f"jrb_{jrb_edition}_index.tsv")
print(song_name_page_nums)

../public/jrb_4.pdf
../data_files/jrb_4_index.tsv


In [28]:
df = pd.read_csv(song_name_page_nums, sep='\t', names=["page_num", "song_name"], quoting=csv.QUOTE_NONE)
df['page_num'] = pd.to_numeric(df['page_num'], errors='coerce').astype(int)
df_sorted = df.sort_values(by='page_num')
df_sorted

,page_num,song_name
0,10,ACROSS THE ALLEY FROM THE ALAMO
1,11,ADIOS
2,12,AFTER THE RAIN
3,14,AGUAS DE MARCO (WATERS OF MARCH)
4,16,ALL NIGHT LONG
...,...,...
395,505,YOU CAME A LONG WAY FROM ST LOUIS
396,506,YOU'RE EVERYTHING
397,508,YOU'RE LOOKING AT ME
398,509,YOU'RE THE CREAM IN MY COFFEE


In [29]:
tot = 0
for i in range(len(df_sorted)):
  current_row = df_sorted.iloc[i]
  page_scans = 1

  if i < len(df_sorted) - 1:
    next_row = df_sorted.iloc[i + 1]
    if not current_row["page_num"] + 1 == next_row["page_num"]:
      page_scans = next_row["page_num"] - current_row["page_num"]
      print(f"{current_row['page_num']:03} - {current_row['song_name']} - {page_scans}")


  tot += 1

print(tot)

012 - AFTER THE RAIN - 2
014 - AGUAS DE MARCO (WATERS OF MARCH) - 2
020 - AND ALL THAT JAZZ - 2
022 - ANGELA - 2
028 - AS WE SPEAK - 2
030 - ASHES TO ASHES - 2
034 - BABY JUST COME HOME TO ME - 2
038 - BAG'S NEW GROOVE - 2
056 - BORN TO BE BAD - 2
058 - BOY MEETS HORN - 2
062 - BROWN HORNET - 2
068 - CAKE WALKING BABIES FROM HOME - 2
070 - THE CAPE VERDEAN BLUES - 2
074 - CHARADE - 2
076 - CHASING THE BIRD - 2
078 - CHILDREN OF THE NIGHT - 2
084 - CHROMOZONE - 2
090 - COME DANCE WITH ME - 2
094 - CURVES AHEAD - 2
100 - DEACON BLUES - 2
104 - DEVIL MAY CARE - 2
122 - EFFENDI - 2
130 - FEDERICO - 2
132 - A FELICIDADE - 2
136 - FLIP, FLOP AND FLY - 2
142 - GETTIN' OVER THE BLUES - 2
146 - GIRL WITH HIS SMILE AND MY EYES - 2
148 - GIVE ME THE NIGHT - 2
164 - HIDEAWAY - 2
168 - HOE-DOWN - 2
170 - HOME - 2
174 - HOW LITTLE WE KNOW - 2
178 - I COULD EAT YOUR WORDS - 2
190 - I SAY A LITTLE PRAYER - 2
194 - I TOLD YA I LOVE YA NOW GET OUT - 2
204 - IF EVER I WOULD LEAVE YOU - 2
206 - IF I RULED

In [ ]:
def extract_and_save_pages(filename, pdf_save_path, start, end):
  doc = pymupdf.open(filename)
  pages_to_keep = list(range(start, end))
  doc.select(pages_to_keep) 
  doc.save(
      pdf_save_path + ".pdf",
      garbage=3, 
      deflate=True, 
      clean=True
  )
  doc.close()

In [35]:
tot = 0
offset = 0 # this has to be computed manually.
# For:
  # jrb_2, offset is 7
  # jrb_3, offset is 5
  # jrb_4, offset if 0
  # jrb_5, offset is 13
  # jrb_6, offset is -1

for i in range(len(df_sorted)):
  current_row = df_sorted.iloc[i]
  page_scans = 1

  if i < len(df_sorted) - 1:
    next_row = df_sorted.iloc[i + 1]
    if not current_row["page_num"] + 1 == next_row["page_num"]:
      page_scans = next_row["page_num"] - current_row["page_num"]
      

  song_folder = os.path.join(jrb_folderpath, str(current_row["page_num"]))
  # print(song_folder)
  # print(f"{current_row['page_num']:03} - {current_row['song_name']} - {page_scans}")
  cleaned_songname = current_row['song_name'].strip().upper().replace(" ", "_")
  # print(cleaned_songname)

  os.makedirs(song_folder, exist_ok=True)
  savename = os.path.join(song_folder, cleaned_songname)
  pdf_start_page = current_row["page_num"] - 1 + offset # subtract 1 for 0 index, add offset
  pdf_end_page = pdf_start_page + page_scans
  if pdf_start_page == pdf_end_page: # happens because 2 songs are on 1 page. 
    pdf_end_page += 1
  # print(pdf_start_page, pdf_end_page, cleaned_songname)
  extract_and_save_pages(jrb_filepath, savename, pdf_start_page, pdf_end_page)

  tot += 1

print(tot)

400


In [36]:
# Extra util to create json files
df_sorted["edition"] = jrb_edition
save_filename = os.path.join("..", "data_files", f"jrb_{jrb_edition}_index.json")
df_sorted.to_json(save_filename, orient="records", indent=2)